In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402
import mne  # noqa: E402
from IPython.display import Image, display  # noqa: E402
from joblib import Parallel, delayed  # noqa: E402
from scipy.signal import butter, hilbert, sosfiltfilt  # noqa: E402
from scipy.stats import zscore  # noqa: E402

# The stage-00 PCA scripts select the Agg backend on import; `%matplotlib inline`
# below restores inline figures.
from scripts import run_assr_raw_pca as raw_pca  # noqa: E402
from scripts import run_assr_wavelet_pca as wavelet_pca  # noqa: E402
from scripts.export_aligned_assr_fif import (  # noqa: E402
    ONSETS_CSV_NAME,
    STIMULUS_ONSET_DESCRIPTION,
)
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.analysis import assr_trials as at  # noqa: E402
from src.analysis.assr_variants import zscore_time  # noqa: E402
from src.analysis.data_representations import (  # noqa: E402
    AnalysisData,
    DataRepresentation,
    to_wavelet_power,
)
from src.analysis.pca_polarity import (  # noqa: E402
    align_pc1_signs,
    apply_pc1_signs,
    topography_consistency,
)
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    CoordinateSystems,
    ExperimentNames,
    MusicTypeVariants,
    PreprocessedDataVariants,
    SingleDataMetadata,
)
from src.io.loading import assr_electrode_mask, concatenated_dir  # noqa: E402
from src.io.wavelet_store import find_wavelet_set  # noqa: E402
from src.preprocessing.pipeline import DatasetHandler  # noqa: E402
from src.preprocessing.stimulus_alignment import (  # noqa: E402
    get_stimulus_onset_samples,
    resolve_stimulus_marker,
)

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
print("Setup complete.")

# Aligned FIF Export — ASSR

Inspects the standalone export of
[`scripts/export_aligned_assr_fif.py`](../../scripts/export_aligned_assr_fif.py)
(PBS: `jobs/metacentrum/00-preprocessing/run_export_aligned_assr_fif.pbs`): every ASSR
recording, no exclusions, cut so that stimulus *k* lands at the same sample in every
file. The export lives outside the analysis workflow; this notebook only reads it.

Six checks, from the cheapest to the most physiological:

1. **Format** — every file has the same length, sampling rate and onset grid, and the
   `stimulus_onset` annotations equal `stimulus_onsets.csv`.
2. **Cuts** — every cut sits exactly `keep_tail` before an onset, and every sample of
   every file is a verbatim copy of the `after_ica` source at its *calibrated* onset
   (marker + per-recording offset, see `src/preprocessing/marker_shift.py`).
3. **Response timing** — the 40 Hz inter-trial phase coherence of every recording
   switches on at 0 s and off near 0.5 s. A recording whose onsets were mis-timed
   would show its response shifted against the others.
4. **Wavelet store** — the export, transformed exactly as the store was, matches the
   store's power trial by trial at the onsets the analyses use.
5. **Raw channel PCA** — stage 00's raw-voltage PCA run on the stored input and on the
   export gives the same components.
6. **Wavelet channel PCA** — stage 00's wavelet PCA run on the store and on the
   export's wavelets.

> **Requires the export to have run**
> (`qsub jobs/metacentrum/00-preprocessing/run_export_aligned_assr_fif.pbs`).

## Configuration

In [ ]:
ASSR_DIR = ProjectPaths.PROCESSED_DATA_DIR / ExperimentNames.ASSR.value
ALIGNED_DIR = ASSR_DIR / "aligned_fif"
# What the export read (its --input_dir).
SOURCE_DIR = ASSR_DIR / PreprocessedDataVariants.RAW_AFTER_ICA.value
KEEP_TAIL_SEC = AssrEpoch.PRE_ONSET_S  # the export's --keep_tail_sec

# None = all recordings; a small number for a quick look (the checks of section 2
# read every file and its source in full).
N_RECORDINGS: int | None = None

# ── Response timing ───────────────────────────────────────────────────────────
ASSR_BAND = (35.0, 45.0)  # Hz, around the 40 Hz stimulation rate
EPOCH_TMIN, EPOCH_TMAX = -AssrEpoch.PRE_ONSET_S, AssrEpoch.POST_ONSET_S
STIMULUS_END_S = AssrEpoch.STIMULUS_DURATION_S
# Flag a recording lagging the cohort by more than this. Subjects' ASSR latencies
# differ by up to ~20 ms; the errors to catch are the uncalibrated marker shift
# (370-455 ms) and the fallback offset's residual (up to ~50 ms).
LATENCY_OUTLIER_MS = 30.0

# ── Wavelet store comparison and channel PCA (sections 4-6) ───────────────────
WAVELET_DIR = ASSR_DIR / "wavelets"
CONDITIONS = (ConditionVariants.PLACEBO, ConditionVariants.PSILOCYBIN)
# Recordings per condition, in participant-ID order: of the stored ones and of the
# ones the store leaves out. None = all.
N_STORE_RECORDINGS: int | None = None
MAX_STORE_LAG_MS = 48.0  # search range of the per-trial lag
# Channels the section-4 power and the wavelet PCA cover: "all" (the whole montage,
# as the wavelet PCA job) or "roi" (the ASSR electrodes of config/assr_electrodes/).
TF_CHANNELS = "all"
WAVELET_CHUNK = 16  # channels transformed / read per step (memory bound)
# Recordings transformed in parallel. Over the whole montage one recording costs
# about four minutes of one core, so N_JOBS divides section 4's runtime.
N_JOBS = 4
N_PCA_COMPONENTS = 3  # the PCA jobs' default

# ── Plot saving ───────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR / "00-preprocessing" / "plots" / "aligned_fif_inspection"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Aligned export : {ALIGNED_DIR}")
print(f"Source         : {SOURCE_DIR}")
print(f"Plots          -> {PLOTS_DIR}")

## 1. Format and Onset Grid

Headers only (`preload=False`). Every row should share `n_times`, `sfreq` and
`n_onsets`, and `onsets_match_csv` should be true everywhere.

In [ ]:
paths = sorted(ALIGNED_DIR.glob("*.fif"))[:N_RECORDINGS]
onsets_csv = pd.read_csv(ALIGNED_DIR / ONSETS_CSV_NAME)
onsets = onsets_csv["onset_sample"].to_numpy()

rows = []
for path in paths:
    raw = mne.io.read_raw_fif(path, preload=False)
    stored = get_stimulus_onset_samples(raw, STIMULUS_ONSET_DESCRIPTION)
    rows.append(
        {
            "recording": path.stem,
            "n_times": raw.n_times,
            "sfreq": raw.info["sfreq"],
            "n_channels": len(raw.ch_names),
            "n_onsets": len(stored),
            "onsets_match_csv": np.array_equal(stored, onsets),
            "n_cuts": int((raw.annotations.description == "BAD boundary").sum()),
        }
    )
summary = pd.DataFrame(rows)
sfreq = float(summary["sfreq"].iloc[0])
tail = int(round(KEEP_TAIL_SEC * sfreq))

print(f"Recordings              : {len(summary)}")
print(f"Trials (CSV)            : {len(onsets)}")
for column in ["n_times", "sfreq", "n_channels", "n_onsets"]:
    print(f"Same {column:<19}: {summary[column].nunique() == 1}")
print(f"Onsets == CSV everywhere: {summary['onsets_match_csv'].all()}")
print(
    f"Length                  : {summary['n_times'].iloc[0] / sfreq:.2f} s, first "
    f"onset at {onsets[0] / sfreq:.3f} s"
)
summary

In [ ]:
trial_s = np.diff(onsets) / sfreq

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(trial_s, marker="o", ms=3, lw=0.8)
axes[0].axhline(
    EPOCH_TMAX + KEEP_TAIL_SEC,
    color="red",
    ls="--",
    lw=1,
    label="epoch end + keep_tail",
)
axes[0].set_title("Trial length (onset to next onset) — shared by every file")
axes[0].set_xlabel("Trial index")
axes[0].set_ylabel("Length (s)")
axes[0].legend()

axes[1].bar(np.arange(len(summary)), summary["n_cuts"])
axes[1].set_title("Cuts per recording (intervals that were longer than the minimum)")
axes[1].set_xlabel("Recording index")
axes[1].set_ylabel("Cuts")

plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "trial_lengths_and_cuts.png", dpi=150)
plt.show()

# Data is continuous from onset - keep_tail to next onset - keep_tail, so an epoch
# may reach at most (shortest trial - keep_tail) after its onset.
print(
    f"Trial length {trial_s.min():.3f}-{trial_s.max():.3f} s; continuous after each "
    f"onset: >= {trial_s.min() - KEEP_TAIL_SEC:.3f} s "
    f"(epoch needs {EPOCH_TMAX:.3f} s: {trial_s.min() - KEEP_TAIL_SEC >= EPOCH_TMAX})"
)

## 2. Cuts and Verbatim Copy

Each file is read once together with its `after_ica` source. The source onsets are
recomputed here from the `fam+` markers and the per-recording calibration, the same
way the export found them, and the aligned file is covered by windows mapped back
onto the source:

- the lead-in `[0, onset₀)`,
- each trial `[onsetₖ − keep_tail, onsetₖ₊₁ − keep_tail)`,
- the last trial to the end of the file.

Together they cover every sample, so `max_abs_diff = 0` means the file is exactly
the source with only the removed intervals missing. `cuts_at_tail` checks that every
`BAD boundary` lies exactly `keep_tail` before an onset, i.e. never inside a trial.

The same pass keeps each recording's 40 Hz phase per trial on the ASSR electrodes
(`config/assr_electrodes/`) for section 3, so nothing has to be read twice.

In [ ]:
marker = resolve_stimulus_marker(ExperimentNames.ASSR)
sos = butter(4, ASSR_BAND, btype="bandpass", fs=sfreq, output="sos")
pre = int(round(-EPOCH_TMIN * sfreq))
post = int(round(EPOCH_TMAX * sfreq))
epoch_times = np.arange(-pre, post) / sfreq
epoch_index = onsets[:, None] + np.arange(-pre, post)  # (trials, epoch samples)

checks, itc = [], []
for path in paths:
    aligned = mne.io.read_raw_fif(path, preload=True)
    source = mne.io.read_raw_fif(SOURCE_DIR / path.name, preload=True)
    offset = marker.onset_offset_for(path.name)
    source_onsets = get_stimulus_onset_samples(source, marker.label, offset)
    source_onsets = source_onsets[: len(onsets)]

    a, s = aligned.get_data(), source.get_data()
    # (aligned start, aligned stop, source start) covering the whole aligned file.
    windows = [(0, onsets[0], source_onsets[0] - onsets[0])]
    stops = np.append(onsets[1:] - tail, aligned.n_times)
    windows += [
        (on - tail, stop, src - tail)
        for on, stop, src in zip(onsets, stops, source_onsets)
    ]
    max_diff = max(
        np.abs(a[:, start:stop] - s[:, src : src + stop - start]).max()
        for start, stop, src in windows
    )

    is_cut = aligned.annotations.description == "BAD boundary"
    cut_samples = np.round(
        (aligned.annotations.onset[is_cut] - aligned.first_time) * sfreq
    ).astype(int)

    eeg_names = aligned.copy().pick("eeg").ch_names
    roi = assr_electrode_mask(eeg_names, strict=False)
    roi_signal = aligned.get_data(picks="eeg")[roi].mean(axis=0)
    phase = np.angle(hilbert(sosfiltfilt(sos, roi_signal)))
    itc.append(np.abs(np.exp(1j * phase[epoch_index]).mean(axis=0)))

    checks.append(
        {
            "recording": path.stem,
            "onset_offset_s": offset,
            "source_onsets": len(
                get_stimulus_onset_samples(source, marker.label, offset)
            ),
            "max_abs_diff": max_diff,
            "n_cuts": int(is_cut.sum()),
            "cuts_at_tail": bool(np.isin(cut_samples, onsets - tail).all()),
            "roi_channels": int(roi.sum()),
        }
    )
    del aligned, source, a, s

checks = pd.DataFrame(checks)
itc = np.stack(itc)  # (recordings, epoch samples)
print(
    f"Verbatim copy everywhere (max |diff| == 0): {(checks['max_abs_diff'] == 0).all()}"
)
print(f"Every cut keep_tail before an onset      : {checks['cuts_at_tail'].all()}")
checks

## 3. Response Timing

40 Hz inter-trial phase coherence (ITC) per recording: the ROI-mean signal is
band-passed to `ASSR_BAND`, its Hilbert phase epoched around every onset, and the
phases averaged over trials. ITC does not depend on a recording's amplitude, so
every row is comparable. A correctly aligned recording is phase-locked from shortly
after 0 s (the auditory pathway adds a few tens of ms) until shortly after 0.5 s.

The band-pass runs over the cut file, so its ringing at a cut can touch the first
few tens of ms of the pre-onset window; the stimulus window is unaffected.

In [ ]:
fig, axes = plt.subplots(
    2, 1, figsize=(14, 9), sharex=True, gridspec_kw={"height_ratios": [2, 1]}
)
image = axes[0].imshow(
    itc,
    aspect="auto",
    origin="lower",
    extent=[epoch_times[0], epoch_times[-1], -0.5, len(itc) - 0.5],
    cmap="viridis",
)
fig.colorbar(image, ax=axes, location="right", shrink=0.6, label="ITC")
axes[0].set_title(f"{ASSR_BAND[0]:.0f}-{ASSR_BAND[1]:.0f} Hz ITC per recording")
axes[0].set_ylabel("Recording index")

for row in itc:
    axes[1].plot(epoch_times, row, color="0.6", lw=0.6, alpha=0.5)
axes[1].plot(epoch_times, itc.mean(axis=0), color="black", lw=2, label="mean")
axes[1].set_xlabel("Time from onset (s)")
axes[1].set_ylabel("ITC")
axes[1].legend()
for ax in axes:
    ax.axvline(0.0, color="red", ls="--", lw=1)
    ax.axvline(STIMULUS_END_S, color="red", ls="--", lw=1)

if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "itc_per_recording.png", dpi=150, bbox_inches="tight")
plt.show()

### Timing Against the Cohort

Each recording's ITC course (smoothed over `SMOOTH_MS`) is cross-correlated with the
mean of all the *other* recordings over −50 to 750 ms; `lag_ms` is where that
correlation peaks (positive = this recording responds later). Correctly aligned
recordings sit near 0 ms. The marker error the calibration removes was 370-455 ms,
so a mis-timed recording would be off by far more than `LATENCY_OUTLIER_MS`.

A lag is only meaningful when there is a response: with *N* trials, random phases
already give an ITC of about √(π / 4N), so a recording whose stimulus plateau does
not clear `MIN_PLATEAU_RATIO` × that floor is marked `weak`. Only a `reliable`
recording (not weak, peak correlation ≥ `MIN_CORR`) is ever flagged.

In [ ]:
from scipy.signal import correlate, correlation_lags  # noqa: E402

SMOOTH_MS = 20.0
MAX_LAG_MS = 100.0
MIN_PLATEAU_RATIO = 2.0
MIN_CORR = 0.5  # below this the course does not resemble the cohort's

noise_itc = np.sqrt(np.pi / (4 * len(onsets)))
kernel = np.ones(int(round(SMOOTH_MS * 1e-3 * sfreq)))
kernel /= kernel.size
smooth = np.stack([np.convolve(row, kernel, mode="same") for row in itc])

span = (epoch_times >= -0.05) & (epoch_times <= 0.75)
plateau = (epoch_times >= 0.1) & (epoch_times <= 0.4)
lags = correlation_lags(span.sum(), span.sum(), mode="full")
searched = np.abs(lags) <= int(round(MAX_LAG_MS * 1e-3 * sfreq))


def _z(x):
    return (x - x.mean()) / (x.std() + 1e-12)


timing = []
for i, name in enumerate(checks["recording"]):
    others = np.delete(smooth, i, axis=0).mean(axis=0)
    x, y = _z(smooth[i][span]), _z(others[span])
    cc = correlate(x, y, mode="full")[searched] / span.sum()
    best = int(np.argmax(cc))
    plateau_itc = float(np.median(itc[i][plateau]))
    timing.append(
        {
            "recording": name,
            "plateau_itc": plateau_itc,
            "weak": plateau_itc < MIN_PLATEAU_RATIO * noise_itc,
            "lag_ms": 1e3 * lags[searched][best] / sfreq,
            "corr": cc[best],
        }
    )
timing = pd.DataFrame(timing)
timing["reliable"] = ~timing["weak"] & (timing["corr"] >= MIN_CORR)
timing["flagged"] = timing["reliable"] & (timing["lag_ms"].abs() > LATENCY_OUTLIER_MS)

responders = timing[timing["reliable"]]
print(f"Random-phase ITC floor : {noise_itc:.3f} ({len(onsets)} trials)")
print(f"Reliable responders    : {len(responders)} / {len(timing)}")
if len(responders):
    print(
        f"Lag over responders    : median {responders['lag_ms'].median():+.1f} ms, "
        f"range {responders['lag_ms'].min():+.1f} to {responders['lag_ms'].max():+.1f} ms"
    )
print(
    f"Flagged recordings     : {timing.loc[timing['flagged'], 'recording'].tolist() or 'none'}"
)
timing

## 4. Comparison With the Wavelet Store

The ASSR analyses read the wavelet store (`wavelets/<Condition>_ASSR/`) together with
`concatenated/<Condition>_ASSR.stimulus_onsets.npy`. Here every exported recording, of
both conditions, goes through the store's transform — resampled to the store's rate,
`to_wavelet_power` on the store's grid and cycles — and is cut at
`stimulus_onsets.csv` (rescaled); the stored recordings are read from the store and
cut at its onsets file. Power is averaged over the `TF_CHANNELS` channels, by default
the whole montage.

One pass per recording serves sections 4 and 6: each side is transformed (export) or
read (store) once, `WAVELET_CHUNK` channels at a time, and every chunk adds to the
channel means used here and to the per-channel trial averages the wavelet PCA of
section 6 reduces. `N_JOBS` recordings run in parallel; over the whole montage a
recording costs about four minutes of one core.

Per stored recording:

- `r_at_onsets` — correlation of the 35-45 Hz trial courses (0-1 s) with no shift.
  It is the same data, so this is near 1 when the two agree on where each trial is.
- per-trial `lag` — the shift of the store's course that best matches the export's
  trial, within ±`MAX_STORE_LAG_MS` (negative = the store's trial sits earlier than
  its onsets file says). 0 for every trial means the store and its onsets file agree
  with the export; a lag that **drifts with the trial index** means they are on
  different time bases.

Not bit-exact by construction: the store was resampled and transformed on the
continuous recording before cutting, the export is resampled and transformed after,
so the bins near a cut (100 ms before each onset, and the lowest frequencies) differ.
The 40 Hz stimulus window is far from any cut.

In [ ]:
def wavelet_power(data, wavelets):
    """``(C, T)`` signal -> ``(C, F, T)`` power through the store's transform."""
    power = to_wavelet_power(
        AnalysisData(
            data=data[np.newaxis],
            sfreq=wavelets.sfreq,
            representation=DataRepresentation.TIME_DOMAIN,
            label="export",
        ),
        wavelets.freqs,
        n_cycles=wavelets.n_cycles,
        keep_frequency_dim=True,
    ).data[0]
    return power.reshape(data.shape[0], wavelets.n_freqs, -1)


def export_power_chunks(stem, wavelets, names):
    """One exported recording's power, ``(chunk, F, T)`` blocks over *names*."""
    raw = mne.io.read_raw_fif(ALIGNED_DIR / f"{stem}.fif", preload=True)
    data = raw.pick(names).resample(wavelets.sfreq).get_data()
    for start in range(0, len(names), WAVELET_CHUNK):
        yield wavelet_power(data[start : start + WAVELET_CHUNK], wavelets)


def store_power_chunks(wavelets, participant, channel_index):
    """One stored recording's power, ``(chunk, F, T)`` blocks over *channel_index*."""
    array = wavelets.open(participant)
    for start in range(0, len(channel_index), WAVELET_CHUNK):
        yield np.asarray(array[channel_index[start : start + WAVELET_CHUNK]], float)


def tf_readings(zscored, block_onsets, pre, post, baseline_mask):
    """The z-scored channel mean ``(F, T)`` -> both readings' ``(F, W)`` maps.

    "baseline" is the ASSR read-outs' trial normalisation (each trial's own
    pre-onset mean, the pre-onset SD pooled over trials, median over trials);
    "zscored" stops before the pre-onset step: the median over trials only.
    """
    trials, _ = at.cut_trials(zscored, block_onsets, pre, post)
    return {
        "baseline": at.baseline_tf_map(zscored, block_onsets, pre, post, baseline_mask),
        "zscored": np.median(trials, axis=-2),
    }


def recording_pass(stem, participant, wavelets, tf_index, side_onsets, window, band):
    """What sections 4 and 6 need from one recording, one pass over each side.

    Every chunk of channels is z-scored per (channel, frequency) over the recording
    and adds to the plain and the z-scored channel means (section 4); its z-scored
    power averaged over trials is the stack the wavelet PCA reduces (section 6), as
    ``stream_wavelet_pca`` builds it. Only these small results are returned.

    :param participant: The store's participant label, or ``None`` for a recording
        the store does not hold (only the export side is computed).
    :param side_onsets: ``{"export": ..., "store": ...}`` onsets on each time axis.
    """
    names = [wavelets.channel_names[i] for i in tf_index]
    pre, post = -window[0], window[-1] + 1
    sides = {"export": export_power_chunks(stem, wavelets, names)}
    if participant is not None:
        sides["store"] = store_power_chunks(wavelets, participant, tf_index)
    result = {}
    for side, chunks in sides.items():
        plain = zscored = 0.0
        stack = []
        for chunk in chunks:
            z = zscore_time(chunk)
            plain = plain + chunk.sum(axis=0)
            zscored = zscored + z.sum(axis=0)
            stack.append(wavelet_pca.epoch_average(z, side_onsets[side], pre, post)[0])
        result[side] = {
            "tf": tf_readings(
                zscored / len(names), side_onsets[side], pre, post, window < 0
            ),
            "band_course": (plain / len(names))[band].mean(axis=0),  # (T,)
            "stack": np.concatenate(stack),  # (channels, F, W)
        }
    return result


# Every exported recording with its condition and participant, and whether the store
# holds it; participant-ID order within a condition, as the PCA scripts use.
dataset = DatasetHandler(
    ExperimentNames.ASSR, CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS
).dataset_metadata
concat = concatenated_dir(ExperimentNames.ASSR)
wavelet_sets, store_onsets, recordings = {}, {}, []
for condition in CONDITIONS:
    label = f"{condition.value}_{MusicTypeVariants.ASSR.value}"
    wavelet_sets[condition] = find_wavelet_set(WAVELET_DIR / label)
    store_onsets[condition] = np.load(
        concat / f"{label}{ProjectPaths.STIMULUS_ONSETS_SUFFIX}"
    ).astype(int)
    store_meta = pd.read_csv(concat / f"{label}.metadata.csv")
    stored_stems = {Path(f).stem for f in store_meta[str(SingleDataMetadata.FILENAME)]}
    group = dataset[dataset[SingleDataMetadata.CONDITION] == condition]
    group = group.sort_values(SingleDataMetadata.PARTICIPANT_ID)
    entries = [
        (Path(f).stem, str(p).zfill(3))
        for f, p in zip(
            group[SingleDataMetadata.FILENAME], group[SingleDataMetadata.PARTICIPANT_ID]
        )
    ]
    for is_stored in (True, False):
        chosen = [e for e in entries if (e[0] in stored_stems) == is_stored]
        for stem, participant in chosen[:N_STORE_RECORDINGS]:
            recordings.append(
                {
                    "condition": condition,
                    "stem": stem,
                    "participant": participant,
                    "stored": is_stored,
                }
            )

# The two conditions' stores share channels, grid and rate.
wavelets = wavelet_sets[CONDITIONS[0]]
if TF_CHANNELS == "roi":
    keep = assr_electrode_mask(wavelets.channel_names, strict=False)
else:
    keep = np.ones(wavelets.n_channels, dtype=bool)
tf_index = np.flatnonzero(keep)
tf_names = [wavelets.channel_names[i] for i in tf_index]
band = (wavelets.freqs >= ASSR_BAND[0]) & (wavelets.freqs <= ASSR_BAND[1])
window = np.arange(
    int(round(EPOCH_TMIN * wavelets.sfreq)), int(round(EPOCH_TMAX * wavelets.sfreq))
)
stimulus = window >= 0
export_onsets_w = np.round(onsets * wavelets.sfreq / sfreq).astype(int)

n_stored = sum(r["stored"] for r in recordings)
print(
    f"{len(recordings)} recordings ({n_stored} stored), {len(tf_names)} channels "
    f"({TF_CHANNELS}), {N_JOBS} in parallel"
)
outputs = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(recording_pass)(
        r["stem"],
        r["participant"] if r["stored"] else None,
        wavelet_sets[r["condition"]],
        tf_index,
        {"export": export_onsets_w, "store": store_onsets[r["condition"]]},
        window,
        band,
    )
    for r in recordings
)
for r, output in zip(recordings, outputs):
    r.update(output)

# Trial matching, on the plain channel-mean power as the store holds it.
max_lag = int(round(MAX_STORE_LAG_MS * 1e-3 * wavelets.sfreq))
shifts = np.arange(-max_lag, max_lag + 1)
store_rows = []
for r in recordings:
    if not r["stored"]:
        continue
    condition_onsets = store_onsets[r["condition"]]
    export_band = r["export"]["band_course"][export_onsets_w[:, None] + window]
    store_course = r["store"]["band_course"]
    lags = np.empty(len(condition_onsets))
    for k, onset in enumerate(condition_onsets):
        fits = [
            np.corrcoef(
                export_band[k, stimulus], store_course[onset + shift + window[stimulus]]
            )[0, 1]
            for shift in shifts
        ]
        lags[k] = 1e3 * shifts[int(np.argmax(fits))] / wavelets.sfreq
    r["lags"] = lags
    store_trials = store_course[condition_onsets[:, None] + window]
    store_rows.append(
        {
            "condition": r["condition"].value,
            "participant": r["participant"],
            "recording": r["stem"],
            "r_at_onsets": np.corrcoef(
                export_band[:, stimulus].ravel(), store_trials[:, stimulus].ravel()
            )[0, 1],
            "lag_first_20_ms": np.median(lags[:20]),
            "lag_last_20_ms": np.median(lags[-20:]),
            "power_ratio": np.median(store_course)
            / np.median(r["export"]["band_course"]),
        }
    )
store_cmp = pd.DataFrame(store_rows)
display(
    store_cmp.groupby("condition")[
        ["r_at_onsets", "lag_first_20_ms", "lag_last_20_ms", "power_ratio"]
    ].median()
)
store_cmp

In [ ]:
colors = dict(zip(CONDITIONS, ("tab:blue", "tab:orange")))
fig, ax = plt.subplots(figsize=(14, 4))
for condition in CONDITIONS:
    lags = np.stack(
        [r["lags"] for r in recordings if r["stored"] and r["condition"] == condition]
    )
    for row in lags:
        ax.plot(row, color=colors[condition], lw=0.5, alpha=0.25)
    ax.plot(
        np.median(lags, axis=0),
        color=colors[condition],
        lw=2,
        label=f"{condition.value} median (n={len(lags)})",
    )
ax.axhline(0.0, color="red", ls="--", lw=1)
ax.set_title(
    "Store trial vs export trial: best-matching shift of the store's 35-45 Hz course"
)
ax.set_xlabel("Trial index")
ax.set_ylabel("Lag (ms), negative = store earlier")
ax.legend()
plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "store_lag_per_trial.png", dpi=150)
plt.show()

### Onset TF Maps: Export vs Store

The ASSR read-outs' TF reading (stage 06, §7), on both sides: every (channel,
frequency) of the recording z-scored over time, the mean over the channels,
then every trial referenced to its own pre-onset mean and divided by the pre-onset SD
pooled over all trials, and the median over trials (`assr_trials.baseline_tf_map`).
The maps are in baseline SD; the grand mean over the compared recordings is shown.

Two differences are expected by construction, not from the alignment:

- **The export's pre-onset window starts at a cut.** A wavelet computed across a cut
  carries an artifact as wide as the wavelet (±250 ms at 40 Hz with
  `n_cycles = f / 2`), and here it lands in the baseline itself, inflating the
  baseline SD and so shrinking the export's map. Any TF work on these files should
  keep the store's transform-then-cut order, or export with a longer
  `--keep_tail_sec`.
- **The store is cut at its onsets file**, which section 4 shows drifting against the
  store's trials by up to tens of ms, so part of the stimulus enters the late trials'
  baseline.

In [ ]:
times_tf = window / wavelets.sfreq


def plot_tf_pair(first, second, unit, title, filename):
    """``first | second | second - first`` for two ``(title, per-recording maps)``,
    each the grand mean over its recordings, on one colour scale."""
    (first_title, first_maps), (second_title, second_maps) = first, second
    a = np.mean(np.stack(first_maps), axis=0)
    b = np.mean(np.stack(second_maps), axis=0)
    limit = max(np.abs(a).max(), np.abs(b).max())
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharey=True)
    panels = [
        (f"{first_title}, n={len(first_maps)}", a),
        (f"{second_title}, n={len(second_maps)}", b),
        (f"{second_title} - {first_title}", b - a),
    ]
    for ax, (panel_title, image) in zip(axes, panels):
        mesh = ax.pcolormesh(
            times_tf,
            wavelets.freqs,
            image,
            cmap="RdBu_r",
            vmin=-limit,
            vmax=limit,
            shading="auto",
        )
        ax.set_title(f"{panel_title} ({unit})")
        ax.set_xlabel("Time from onset (s)")
        for edge in (0.0, STIMULUS_END_S):
            ax.axvline(edge, color="black", ls="--", lw=0.8)
    axes[0].set_ylabel("Frequency (Hz)")
    fig.colorbar(mesh, ax=axes, shrink=0.8, label=unit)
    fig.suptitle(title, y=1.02)
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()


def tf_maps_of(condition, side, reading, stored=True):
    return [
        r[side]["tf"][reading]
        for r in recordings
        if r["condition"] == condition and r["stored"] == stored
    ]


def plot_tf_comparison(reading, unit):
    for condition in CONDITIONS:
        plot_tf_pair(
            ("export", tf_maps_of(condition, "export", reading)),
            ("store", tf_maps_of(condition, "store", reading)),
            unit,
            f"{condition.value}: export vs store",
            f"store_vs_export_tf_{reading}_{condition.value}.png",
        )


plot_tf_comparison("baseline", "baseline SD")

### Onset TF Maps Without the Pre-Onset Step

The same chain stopped before the trial normalisation: every (channel, frequency)
z-scored over the recording, the ROI mean, the median over trials. Zero is the
recording's mean level and the unit its SD over the whole recording, so nothing here
depends on the pre-onset window — the cut artifact shows only where it is, just
before 0 s.

In [ ]:
plot_tf_comparison("zscored", "recording SD")

### The Export's Extra Recordings

The export keeps every recording; the store holds only the ones the analyses select,
so the recordings it leaves out (excluded in `data/excluded_participants/assr.csv`,
typically for artifacts) exist only in the export. Per condition, their export maps in
both readings against the export maps of the stored recordings: if the extra
recordings differ, it shows in the difference panel, and the export-vs-store
comparison above is then about the stored ones only.

In [ ]:
extra = [r for r in recordings if not r["stored"]]
print(f"Export recordings not in the store ({len(extra)}):")
for r in extra:
    print(f"  {r['condition'].value:<10} {r['stem']}")

for reading, unit in (("baseline", "baseline SD"), ("zscored", "recording SD")):
    for condition in CONDITIONS:
        extra_maps = tf_maps_of(condition, "export", reading, stored=False)
        if not extra_maps:
            print(f"{condition.value}: every exported recording is stored.")
            continue
        plot_tf_pair(
            ("export, stored", tf_maps_of(condition, "export", reading)),
            ("export, not stored", extra_maps),
            unit,
            f"{condition.value}: the export's extra recordings",
            f"export_extra_vs_stored_tf_{reading}_{condition.value}.png",
        )

## 5. Raw Channel PCA: Stored Input vs Export

Stage 00's raw-voltage PCA ([`scripts/run_assr_raw_pca.py`](../../scripts/run_assr_raw_pca.py))
run twice per condition through the script's own functions and defaults: on the input
the job reads,
`concatenated/<Condition>_ASSR.npy` at its onsets file, and on the export of the same
recordings, resampled to the job's 250 Hz, at `stimulus_onsets.csv`. Each run: every
channel z-scored over the recording, the trial average, a channel PCA per participant,
each component's polarity aligned across participants, and the script's component
comparison (40 Hz SNR, topography consistency). The job writes figures only, so the
stored side is recomputed here.

- `input max |diff|` checks the inputs themselves: the concatenated array is built from
  the same 1000 Hz cut as the export, so it should be 0.
- `r_map` / `r_topo` compare each participant's component (time course, scalp
  loading) between the two runs; `explained` is the variance each run's component
  explains. Any difference left comes from the onsets: the onsets file and the
  export's onsets rounded to 250 Hz differ by up to one sample.

In [ ]:
# Shared by sections 5 and 6: the stored recordings as both PCA scripts read them.
pid_col = str(SingleDataMetadata.PARTICIPANT_ID)
index_col = str(SingleDataMetadata.CONCATENATED_PERSON_INDEX)
file_col = str(SingleDataMetadata.FILENAME)
channel_names = wavelets.channel_names  # the scripts' channel order
raw_args = raw_pca._build_arg_parser().parse_args(
    ["--n_components", str(N_PCA_COMPONENTS)]
)
pca_sfreq = raw_args.sfreq
if pca_sfreq != wavelets.sfreq:
    raise ValueError("The PCA scripts no longer run at the store's rate.")
pca_times = window / pca_sfreq
pca_stim_mask = AssrEpoch.stimulus_mask(pca_times)
pc_labels = [f"PC{c + 1}" for c in range(N_PCA_COMPONENTS)]


def pca_selection(condition):
    """The condition's stored recordings in participant-ID order: both scripts sort
    before the polarity alignment, whose template is seeded from the first subject."""
    label = f"{condition.value}_{MusicTypeVariants.ASSR.value}"
    meta = pd.read_csv(
        concat / f"{label}.metadata.csv", index_col=0, dtype={pid_col: str}
    )
    return meta.sort_values(pid_col).head(N_STORE_RECORDINGS)


for condition in CONDITIONS:
    # Both scripts size the epoch from the onsets file; sections 4-6 share it.
    gaps = np.diff(store_onsets[condition])
    pre = int(round(raw_args.pre_pad * pca_sfreq))
    post = min(int(round(raw_args.post_window * pca_sfreq)), int(gaps.min()))
    if (pre, post) != (-window[0], window[-1] + 1):
        raise ValueError(
            f"{condition.value}: the scripts' epoch differs from section 4's."
        )
    shift = export_onsets_w - store_onsets[condition]
    print(
        f"{condition.value}: export vs stored onsets differ at {np.count_nonzero(shift)}"
        f" of {len(shift)} trials, by {shift.min():+d} to {shift.max():+d} samples"
    )


def export_signal(stem):
    """An exported recording as the scripts' input: their channels, their rate."""
    raw = mne.io.read_raw_fif(ALIGNED_DIR / f"{stem}.fif", preload=True)
    return raw.pick(channel_names).resample(pca_sfreq).get_data()


def align_components(loadings, maps, names):
    """Both scripts' polarity step, per component, in place; the consistencies."""
    consistency = []
    for comp in range(len(loadings)):
        signs, _anchor = align_pc1_signs(loadings[comp], channel_names=names)
        loadings[comp] = apply_pc1_signs(loadings[comp], signs)
        maps[comp] = apply_pc1_signs(maps[comp], signs)
        consistency.append(topography_consistency(loadings[comp]))
    return consistency


def run_agreement(runs, map_key, labels):
    """Per participant and component: how far the export run reproduces the stored."""
    stored, export = runs["stored"], runs["export"]
    rows = []
    for comp, pc in enumerate(pc_labels):
        for subj, label in enumerate(labels):
            rows.append(
                {
                    "component": pc,
                    "participant": label,
                    "r_map": np.corrcoef(
                        stored[map_key][comp, subj].ravel(),
                        export[map_key][comp, subj].ravel(),
                    )[0, 1],
                    "r_topo": np.corrcoef(
                        stored["loadings"][comp, subj], export["loadings"][comp, subj]
                    )[0, 1],
                    "explained_stored": stored["explained"][comp, subj],
                    "explained_export": export["explained"][comp, subj],
                }
            )
    return pd.DataFrame(rows)


def show_agreement(runs, map_key, labels):
    agreement = run_agreement(runs, map_key, labels)
    display(
        agreement.groupby("component")[
            ["r_map", "r_topo", "explained_stored", "explained_export"]
        ].agg(["median", "min"])
    )
    display(pd.concat({n: runs[n]["comparison"] for n in ("stored", "export")}, axis=1))

In [ ]:
def raw_pca_run(signals, pca_onsets, n_subj):
    """run_assr_raw_pca's numerics on one input; *signals* in participant order."""
    pre, post = -window[0], window[-1] + 1
    time_courses = np.zeros((N_PCA_COMPONENTS, n_subj, len(window)))
    loadings = np.zeros((N_PCA_COMPONENTS, n_subj, len(channel_names)))
    explained = np.zeros((N_PCA_COMPONENTS, n_subj))
    for subj, signal in enumerate(signals):
        if raw_args.zscore_per_channel:
            signal = zscore(signal, axis=1)
        evoked, _ = raw_pca.epoch_average(signal, pca_onsets, pre, post)
        (
            time_courses[:, subj],
            loadings[:, subj],
            explained[:, subj],
        ) = raw_pca.channel_pca_components(evoked, N_PCA_COMPONENTS)
    consistency = align_components(loadings, time_courses, channel_names)
    comparison = raw_pca.component_comparison(
        time_courses,
        explained,
        consistency,
        pc_labels,
        pca_stim_mask,
        pca_sfreq,
        raw_args.assr_freq,
        raw_args.snr_threshold,
    )
    return {
        "time_courses": time_courses,
        "loadings": loadings,
        "explained": explained,
        "consistency": consistency,
        "comparison": comparison,
    }


def stored_signals(array, meta):
    for index in meta[index_col]:
        yield np.asarray(array[int(index)], dtype=float)


def export_signals(array, meta, input_diff):
    for index, filename in zip(meta[index_col], meta[file_col]):
        signal = export_signal(Path(filename).stem)
        stored = np.asarray(array[int(index)], dtype=float)
        same = signal.shape == stored.shape
        input_diff.append(np.abs(signal - stored).max() if same else np.inf)
        yield signal


raw_runs = {}
for condition in CONDITIONS:
    meta = pca_selection(condition)
    label = f"{condition.value}_{MusicTypeVariants.ASSR.value}"
    array = np.load(concat / f"{label}.npy", mmap_mode="r")
    input_diff = []
    raw_runs[condition] = {
        "stored": raw_pca_run(
            stored_signals(array, meta), store_onsets[condition], len(meta)
        ),
        "export": raw_pca_run(
            export_signals(array, meta, input_diff), export_onsets_w, len(meta)
        ),
        "labels": [f"PSI{pid}" for pid in meta[pid_col]],
    }
    print(
        f"=== {condition.value}: input max |export - stored| = {max(input_diff):.3g} "
        f"over {len(input_diff)} recordings"
    )
    show_agreement(raw_runs[condition], "time_courses", raw_runs[condition]["labels"])

In [ ]:
# Group-mean component time courses of the two runs: components x conditions.
fig, axes = plt.subplots(
    N_PCA_COMPONENTS,
    len(CONDITIONS),
    figsize=(7 * len(CONDITIONS), 3 * N_PCA_COMPONENTS),
    sharex=True,
    squeeze=False,
)
styles = {
    "stored": dict(color="black", lw=2),
    "export": dict(color="red", lw=1.2, ls="--"),
}
for col, condition in enumerate(CONDITIONS):
    for comp, pc in enumerate(pc_labels):
        ax = axes[comp, col]
        for name, style in styles.items():
            course = raw_runs[condition][name]["time_courses"][comp].mean(axis=0)
            ax.plot(pca_times, course, label=name, **style)
        ax.axvspan(0.0, STIMULUS_END_S, color="grey", alpha=0.12, lw=0)
        ax.set_ylabel(f"{pc} (z-scored)")
    n_subj = len(raw_runs[condition]["labels"])
    axes[0, col].set_title(f"{condition.value} (n={n_subj})")
    axes[-1, col].set_xlabel("Time from onset (s)")
axes[0, 0].legend()
fig.suptitle("Raw channel PCA, group mean: stored input vs export", y=1.01)
plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "raw_pca_stored_vs_export.png", dpi=150)
plt.show()

# The job's own overlay figure, drawn by its plotter for each condition and run.
for condition in CONDITIONS:
    for name in ("stored", "export"):
        run = raw_runs[condition][name]
        run_dir = PLOTS_DIR / "raw_pca" / condition.value / name
        run_dir.mkdir(parents=True, exist_ok=True)
        raw_pca.plot_timecourse_overlay(
            run["time_courses"],
            run["explained"],
            run["consistency"],
            pca_times,
            raw_runs[condition]["labels"],
            "z-scored",
            pc_labels,
            STIMULUS_END_S,
            f"{condition.value}/ASSR (n={len(raw_runs[condition]['labels'])}), {name}",
            run_dir,
        )
        display(Image(filename=str(run_dir / "raw_pca_timecourse_overlay.png")))

## 6. Wavelet Channel PCA: Stored Wavelets vs Export

Stage 00's wavelet PCA ([`scripts/run_assr_wavelet_pca.py`](../../scripts/run_assr_wavelet_pca.py))
per condition, on the store at its onsets file (what the job reads) and on the export's
wavelets at `stimulus_onsets.csv`. Its per-participant input — every channel's power
z-scored per frequency over the recording, then averaged over trials — was built in
section 4's pass exactly as the script's `stream_wavelet_pca` builds it, so no data is
read again here. Then the script's own steps: `channel_pca_tf_maps` per participant,
each component's polarity aligned across participants, and its component comparison
(ASSR-band and broadband driven-minus-baseline contrast). With `TF_CHANNELS = "roi"`
the PCA covers only the ASSR electrodes and is no longer the job's.

Unlike section 5 the two inputs are not the same numbers, for the two reasons shown
in section 4: the store's trials drift against its onsets file, and the export's
wavelets carry the cut artifact before every onset. The job's own group figure for
each condition is shown with both runs when it is on disk.

In [ ]:
wavelet_args = wavelet_pca._build_arg_parser().parse_args(
    ["--n_components", str(N_PCA_COMPONENTS)]
)
if (wavelet_args.sfreq, wavelet_args.pre_pad, wavelet_args.post_window) != (
    raw_args.sfreq,
    raw_args.pre_pad,
    raw_args.post_window,
):
    raise ValueError("The raw and wavelet PCA scripts no longer share an epoch.")


def wavelet_pca_run(stacks):
    """run_assr_wavelet_pca's numerics on per-participant ``(C, F, W)`` stacks."""
    n_subj = len(stacks)
    tf_maps = np.zeros((N_PCA_COMPONENTS, n_subj, wavelets.n_freqs, len(window)))
    loadings = np.zeros((N_PCA_COMPONENTS, n_subj, len(tf_names)))
    explained = np.zeros((N_PCA_COMPONENTS, n_subj))
    for subj, stack in enumerate(stacks):
        (
            tf_maps[:, subj],
            loadings[:, subj],
            explained[:, subj],
        ) = wavelet_pca.channel_pca_tf_maps(stack, N_PCA_COMPONENTS)
    consistency = align_components(loadings, tf_maps, tf_names)
    comparison, _band = wavelet_pca.component_comparison(
        tf_maps,
        explained,
        consistency,
        pc_labels,
        wavelets.freqs,
        pca_stim_mask,
        window < 0,
        wavelet_args.assr_freq,
        wavelet_args.assr_halfwidth,
    )
    return {
        "tf_maps": tf_maps,
        "loadings": loadings,
        "explained": explained,
        "consistency": consistency,
        "comparison": comparison,
    }


wavelet_runs = {}
for condition in CONDITIONS:
    stored = [r for r in recordings if r["stored"] and r["condition"] == condition]
    wavelet_runs[condition] = {
        "stored": wavelet_pca_run([r["store"]["stack"] for r in stored]),
        "export": wavelet_pca_run([r["export"]["stack"] for r in stored]),
        "labels": [f"PSI{r['participant']}" for r in stored],
    }
    print(f"=== {condition.value}")
    show_agreement(
        wavelet_runs[condition], "tf_maps", wavelet_runs[condition]["labels"]
    )

In [ ]:
GROUP_FIGURE = "wavelet_pca_tf_group_average.png"

for condition in CONDITIONS:
    label = f"{condition.value}_{MusicTypeVariants.ASSR.value}"
    job_figure = (
        ProjectPaths.analysis_plots_dir(
            "00-preprocessing", label, analysis_type="assr_wavelet_pca"
        )
        / GROUP_FIGURE
    )
    if job_figure.exists():
        print(f"{condition.value}, the job's figure: {job_figure}")
        display(Image(filename=str(job_figure)))
    else:
        print(f"{condition.value}: no job figure at {job_figure}.")

    # Each run drawn by the job's own plotter.
    n_subj = len(wavelet_runs[condition]["labels"])
    for name in ("stored", "export"):
        run = wavelet_runs[condition][name]
        run_dir = PLOTS_DIR / "wavelet_pca" / condition.value / name
        run_dir.mkdir(parents=True, exist_ok=True)
        wavelet_pca.plot_pca_group(
            run["tf_maps"],
            run["explained"],
            run["consistency"],
            pca_times,
            wavelets.freqs,
            wavelet_args.assr_freq,
            pc_labels,
            f"{condition.value}/ASSR (n={n_subj}), {name}",
            run_dir,
        )
        print(f"{condition.value}, this notebook, {name}:")
        display(Image(filename=str(run_dir / GROUP_FIGURE)))

## What to Look For

- **Section 1**: one value per column across all files, and `onsets_match_csv` true
  everywhere.
- **Section 2**: `max_abs_diff == 0` and `cuts_at_tail` true for every recording.
  Anything else means the export does not match the source at the calibrated onsets.
- **Section 3**: vertical on/off edges at the same latency in every row, and no
  flagged recording. A recording that is not `reliable` is a poor responder, not
  evidence either way. A flagged one is worth checking against its calibration entry in
  `config/participant_mappings/assr_time_shift.csv`.
- **Section 4**: `r_at_onsets` near 1 and a lag at 0 ms for every trial. A lag that
  drifts with the trial index while `r` stays high means the store and its onsets
  file describe different time bases, not that the export is wrong — sections 1-3
  check the export on its own.
- **Section 5**: input difference 0, and `r_map` / `r_topo` near 1 with matching
  explained variance for every participant and component: the raw PCA cannot tell
  the export from its own input.
- **Section 6**: the same structure in both runs' group maps and comparison tables.
  Per-participant agreement is lower than in section 5 by construction (the store's
  drifting onsets, the cut artifact in the export's wavelets), so read it against
  those two, not against 1.